# 16. LSTM — choosing the settings (Method #2: the grid search)
In class, `GridSearchCV(estimator, param_grid, scoring, cv=5)` tried every combination of settings and kept the best (`best_params_`). This notebook shows how the LSTM's settings were chosen in the same spirit — and why `GridSearchCV` itself could not be used.

* **Needs:** `results/lstm_tuning.csv` (the saved search). **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

## Why not `GridSearchCV`?
`cv=5` cuts the data into 5 **random** parts and tests on each in turn — for time series that again means training on the future (notebook 07). So the search was done by hand, with **time-ordered** validation:

* **grid:** look-back {22, 66} days × units {32, 64} × layers {1, 2} = **8 settings**;
* each setting trained for the **first two test years only** (2007 and 2008; validation years 2006 and 2007), with 2 seeds → 32 models;
* **score:** the average validation loss (lower is better);
* the best setting is then **frozen** for all years — the later years never influence the choice.

Training 32 models takes about 15 minutes, so we load the saved search.

In [2]:
from pandas import read_csv
tuning = read_csv(folder + 'results/lstm_tuning.csv')
print(tuning.shape)
tuning.head(8)

(32, 7)


,lookback,hidden,layers,year,seed,val_loss,epochs
0,22,32,1,2007,101,0.911897,21
1,22,32,1,2007,102,0.912382,30
2,22,32,1,2008,101,1.876809,29
3,22,32,1,2008,102,1.835811,31
4,22,32,2,2007,101,0.942739,19
5,22,32,2,2007,102,0.912142,26
6,22,32,2,2008,101,1.782260,31
7,22,32,2,2008,102,1.815914,27


Each row is one trained model. The average validation loss of each setting (`groupby` the three settings):

In [3]:
grid_scores = tuning.groupby(['lookback', 'hidden', 'layers'])['val_loss'].mean().sort_values()
grid_scores.round(3)

lookback  hidden  layers
22        64      2         1.307
66        64      1         1.324
                  2         1.350
22        64      1         1.357
          32      2         1.363
                  1         1.384
66        32      2         1.384
                  1         1.432
Name: val_loss, dtype: float64

In [4]:
best_params = grid_scores.index[0]
print('best_params: lookback =', best_params[0], '| units =', best_params[1], '| layers =', best_params[2])
print('best_score (validation loss) =', round(grid_scores.iloc[0], 3))

best_params: lookback = 22 | units = 64 | layers = 2
best_score (validation loss) = 1.307


**22 days, 64 units, 2 layers** — the setting used in notebook 15 and for every year.

All scores are above 1.0, i.e. worse than simply predicting the average: the validation years 2006 and 2007 (the start of the crisis) look very different from the calm 2003–05 training years. That is the same problem the 'learning from history' question (notebook 19) is about.